# Error Handling in Python

Errors are inevitable. Python's exception system lets you catch them gracefully, give useful feedback, and keep programs running.

**In this notebook:**
- `try / except / else / finally`
- Built-in exception types
- Raising exceptions with `raise`
- Custom exceptions
- Exception chaining
- Common patterns

## 1. try / except

Wrap risky code in `try`. Handle specific exceptions in `except`.

> **Rule:** Always catch the **most specific** exception. Avoid bare `except:` and avoid catching `Exception` for everything.

In [ ]:
# Basic try/except
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: cannot divide by zero")
        return None
    except TypeError:
        print("Error: invalid types")
        return None

print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # None
print(safe_divide("a", 2))   # None

# 'as e' gives access to the exception object
try:
    int("abc")
except ValueError as e:
    print(e)                  # invalid literal for int() with base 10: 'abc'
    print(type(e).__name__)   # ValueError

# Multiple exceptions, same handler
def parse(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0

print(parse("3.14"))   # 3.14
print(parse(None))     # 0.0
print(parse("bad"))    # 0.0

## 2. else and finally

| Clause | Runs when |
|---|---|
| `except` | A matching exception was raised |
| `else` | No exception occurred |
| `finally` | Always — even after `return` or `break` |

In [ ]:
def read_number(text):
    try:
        result = int(text)
    except ValueError as e:
        print(f"  except: {e}")
        result = None
    else:
        # only runs when NO exception occurred
        print(f"  else: parsed successfully → {result}")
    finally:
        # ALWAYS runs
        print(f"  finally: done processing '{text}'")
    return result

print("--- Good input ---")
read_number("42")

print("\n--- Bad input ---")
read_number("abc")

## 3. Built-in Exception Types

Python has a rich hierarchy. Know the most common ones.

In [ ]:
errors = [
    (lambda: int("abc"),         "ValueError"),
    (lambda: "a" + 1,            "TypeError"),
    (lambda: 1 / 0,              "ZeroDivisionError"),
    (lambda: [1,2,3][99],        "IndexError"),
    (lambda: {"a":1}["b"],       "KeyError"),
    (lambda: None.upper(),       "AttributeError"),
]

for func, name in errors:
    try:
        func()
    except Exception as e:
        print(f"{type(e).__name__:<22} → {e}")

## 4. Raising Exceptions

Use `raise` to signal an error from your own code. Always validate before mutating state.

In [ ]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"Age must be int, got {type(age).__name__}")
    if not 0 <= age <= 150:
        raise ValueError(f"Age must be 0–150, got {age}")
    return age

for value in [25, -1, 200, "old"]:
    try:
        print(f"set_age({value!r}) → {set_age(value)}")
    except (TypeError, ValueError) as e:
        print(f"set_age({value!r}) → {type(e).__name__}: {e}")

# Re-raising — log then re-raise unchanged
def process(data):
    try:
        return int(data)
    except ValueError:
        print(f"[LOG] Bad data: {data!r}")
        raise   # re-raises the original ValueError

try:
    process("oops")
except ValueError as e:
    print(f"Caught after re-raise: {e}")

## 5. Custom Exceptions

Inherit from `Exception` to create domain-specific errors with custom attributes.

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the account balance."""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(
            f"Cannot withdraw R$ {amount:.2f} — balance is R$ {balance:.2f}"
        )


class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance

    def withdraw(self, amount):
        if amount > self._balance:
            raise InsufficientFundsError(self._balance, amount)
        self._balance -= amount
        return amount

    def __str__(self):
        return f"Account[{self.owner}]: R$ {self._balance:.2f}"


acc = BankAccount("Alice", 100)
try:
    acc.withdraw(200)
except InsufficientFundsError as e:
    print(e)
    print(f"  Tried to withdraw: R$ {e.amount:.2f}")
    print(f"  Available balance: R$ {e.balance:.2f}")

## 6. Common Patterns

In [ ]:
# Pattern 1: Input validation loop
def get_int(prompt, min_val=None, max_val=None):
    """Keep asking until the user enters a valid integer in range."""
    while True:
        try:
            value = int(input(prompt))
            if min_val is not None and value < min_val:
                raise ValueError(f"Must be >= {min_val}")
            if max_val is not None and value > max_val:
                raise ValueError(f"Must be <= {max_val}")
            return value
        except ValueError as e:
            print(f"  Invalid: {e}. Try again.")

# Pattern 2: Safe file read
def safe_read(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return None
    except PermissionError:
        print(f"No permission: {path}")
        return None

print(safe_read("missing.txt"))   # None — no crash

# Pattern 3: retry
import random
random.seed(1)

def retry(func, times=3):
    for attempt in range(1, times + 1):
        try:
            return func()
        except Exception as e:
            print(f"  Attempt {attempt} failed: {e}")
    raise RuntimeError(f"All {times} attempts failed")

def flaky():
    if random.random() < 0.7:
        raise ValueError("Random failure")
    return "Success!"

try:
    result = retry(flaky, times=5)
    print(result)
except RuntimeError as e:
    print(e)

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | try/except, else/finally, common exceptions |
| [02-medium.py](exercises/02-medium.py) | Medium | raise, custom exceptions, chaining, retry |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | validated class, safe CSV processor, context manager |

Solutions: [solutions/](solutions/)